# Structured Rag

In [1]:
from openai import OpenAI
openai_client = OpenAI()
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [2]:
# setting up the index

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"}
)
files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields = ["title", "description", "content"],
    keyword_fields = ["filename"]
)
index.fit(chunked_docs)
print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [4]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on teh CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [5]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []
    if instructions:
        messages.append({
            "role": "system",
            "content": "instrucctions"
        })
    messages.append({
        "role": "user",
        "content": user_prompt
    })
    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    return response.output_text

In [6]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [7]:
answer = rag('how do I implement LLM as a judge')

In [8]:
answer

'Implementing a Large Language Model (LLM) as a judge involves creating an evaluation framework where the LLM assesses responses based on specific criteria. Here’s a step-by-step guide based on the provided context:\n\n### Step-by-Step Implementation\n\n#### 1. Install Required Libraries\nUse the following command to install the `evidently` library, which will assist in creating and evaluating the LLM judge.\n\n```bash\npip install evidently\n```\n\n#### 2. Import Necessary Modules\nImport the required libraries in your Python environment, such as Jupyter Notebook or Google Colab.\n\n```python\nimport pandas as pd\nimport numpy as np\n\nfrom evidently import Dataset\nfrom evidently import Report\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n```\n\n#### 3. Set Up OpenAI Credentials\nYou need your OpenAI API key to access the LLM.\n\n```python\nimport os\nos.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n```\n\n#### 4. Create Your Dataset\nDesign a Q&A dataset that co

In [22]:
def llm_structured(user_prompt, output_type, 
                   instructions=None, model="gpt-4o-mini"):
    messages = []
    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })
    messages.append({
        "role": "user",
        "content": user_prompt
    })
    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )
    return response.output_parsed

In [24]:
llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent
)

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

## Structured RAG

In [25]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [26]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [50]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

In [59]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [28]:
answer = rag_structured('how do I do llm evals?')
print(answer.answer[:100])
print(answer.found_answer)

To perform LLM evaluations, you can follow these steps based on the context provided:

1. **Installa
True


In [31]:
answer = rag_structured('how do I install kafka on windows')
print(answer.answer)

The provided context does not include any information on how to install Kafka on Windows. Please refer to the official Kafka documentation or installation guide for that specific information.


In [32]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

## Structured Optional

In [99]:
from typing import Optional

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    
    If the answer to the question wasn't found in the database,
    `answer` is Duh
    """
    
    answer: Optional[str] = None
    found_answer: bool

In [62]:
RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG system\n\nIf the answer to the question wasn't found in the database,\n`answer` is Duh",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [63]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [56]:
answer = rag_structured("how do I install kafka on windows", RAGResponse)

print(answer.answer)
print(answer.found_answer)

The context does not provide any information on how to install Kafka on Windows. Please refer to the official Kafka documentation or relevant guides for installation instructions.
False


In [45]:
answer = rag_structured("how do I install kafka on windows", RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [64]:
answer = rag_structured("how do I install kafka on windows", RAGResponse)

print(answer.answer)
print(answer.found_answer)

Duh
False


In [96]:
from typing import Optional
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    
    """
    answer: Optional[str] = Field(None, description="Answer to the question or DOWEE if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [97]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or DOWEE if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [98]:
for i in range(10):
    answer = rag_structured('how do I install kafka on windows?', RAGResponse)

    print(answer.answer)
    print(answer.found_answer)
    print("++++++++++++++++++++++++++++++++++++")

The documentation does not provide information on how to install Kafka on Windows. You may need to look for Kafka-specific installation guides or resources elsewhere.
False
++++++++++++++++++++++++++++++++++++
The context does not provide information on how to install Kafka on Windows.
False
++++++++++++++++++++++++++++++++++++
The provided context does not contain any information about installing Kafka on Windows.
False
++++++++++++++++++++++++++++++++++++
The context does not provide any information on how to install Kafka on Windows. Please refer to Kafka's official documentation or other resources for installation instructions.
False
++++++++++++++++++++++++++++++++++++
The provided context does not contain information about installing Kafka on Windows. Please refer to the official Kafka documentation or relevant resources for installation instructions.
False
++++++++++++++++++++++++++++++++++++
The provided context does not contain information on installing Kafka on Windows. There

In [100]:
from typing import Optional

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    
    If the answer to the question wasn't found in the database,
    `answer` is Duh
    """
    
    answer: Optional[str] = None
    found_answer: bool

for i in range(10):
    answer = rag_structured('how do I install kafka on windows?', RAGResponse)

    print(answer.answer)
    print(answer.found_answer)

Duh
False
Duh
False
Duh
False
Duh
False
Duh
False
Duh
False
Duh
False
Duh
False
Duh
False
Duh
False


** I wonder why the prompt in documentation produces Duh while the description in the Field, does not produce the random word like Duh. Unless I put it None. I'm testing out other words aside from None because None seems to be easy and I wonder if None is like a reserve word or an arbitrary word.

In [106]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [122]:
# TODO: where did Literal came from - how-to, explanation, troubleshooting ...

In [107]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [108]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [114]:
print(answer.followup_questions)

['What tools are available for LLM evaluation?', 'Are there specific metrics recommended for LLM evaluation?', 'Can I evaluate a single LLM without multiple judges?']


** What I'm noticing about pydantic BaseModel, and adding all these description in schema
is that we can just write english in description and llm knows what we mean and organizes the returned data in this way. wait.lets back track. ohh it all boils down to the openai api property, text_format.

_TODO: Need to check if this is available or I can redo this project to local models

In [115]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [116]:
answer

RAGResponse(answer='The documentation does not provide information on how to install Kafka on Windows. If you need help with Kafka installation, I recommend checking the official Kafka documentation or relevant resources on the topic.', found_answer=False, confidence=0.9, confidence_explanation='The content provided does not mention Kafka at all, therefore I cannot provide installation instructions based on the given context.', answer_type='reference', followup_questions=['Where can I find Kafka installation instructions?', 'What are the system requirements for Kafka on Windows?', 'Can you provide links to tutorials on Kafka?'])

## Nested Fields

In [117]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """
    
    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None
        
    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found' not both.")
        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")
        return self

In [118]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [119]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not contain any information about installing Kafka on Windows.'), found_answer=False, answer=None)

In [120]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='### To run LLM evaluations, follow these steps:\n\n1. **Connect to Evidently Cloud**:\n   - Before running evaluations, ensure that you have connected to [Evidently Cloud](https://app.evidently.cloud) and created a project.\n\n2. **Prepare the Dataset**:\n   - Create a toy dataset with your questions and reference answers. Here’s a sample code snippet:\n     ```python\n     data = [\n         ["Why is the sky blue?", "The sky is blue because molecules in the air scatter blue light from the sun more than they scatter red light."],\n         ["How do airplanes stay in the air?", "Airplanes stay in the air because their wings create lift by forcing air to move faster over the top of the wing than underneath."]\n     ]\n     columns = ["question", "target_response"]\n     ref_data = pd.DataFrame(data, columns=columns)\n     ```\n\n3. **Create and Run the Report**:\n   - Use the following code to create a re

In [121]:
from pydantic import ValidationError

try: 
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
